# 🏆 Bolão Copa do Mundo 2026

**EUA · México · Canadá | 11 de junho – 19 de julho de 2026**

---

## Formato do torneio
- **48 seleções** divididas em **12 grupos** de 4 times
- Os **2 primeiros** de cada grupo + os **8 melhores terceiros** avançam para o **Round of 32**
- **104 jogos** no total (72 na fase de grupos + 32 na fase eliminatória)

## Regras do Bolão

| Acerto | Pontos |
|---|---|
| Placar exato | **3 pts** |
| Resultado correto (V/E/D) | **1 pt** |
| Erro | **0 pts** |

> **Perspectiva analítica:** cada jogo exibe as probabilidades do modelo Bayesiano Dixon-Coles e o placar mais provável segundo as simulações (100 mil iterações).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')

In [ ]:
SCORE_COLS_MAP: dict[str, tuple[int, int]] = {
    'zero_zero': (0, 0), 'zero_one': (0, 1), 'zero_two': (0, 2), 'zero_three': (0, 3), 'zero_four': (0, 4),
    'one_zero': (1, 0), 'one_one': (1, 1), 'one_two': (1, 2), 'one_three': (1, 3), 'one_four': (1, 4),
    'two_zero': (2, 0), 'two_one': (2, 1), 'two_two': (2, 2), 'two_three': (2, 3), 'two_four': (2, 4),
    'three_zero': (3, 0), 'three_one': (3, 1), 'three_two': (3, 2), 'three_three': (3, 3), 'three_four': (3, 4),
    'four_zero': (4, 0), 'four_one': (4, 1), 'four_two': (4, 2), 'four_three': (4, 3), 'four_four': (4, 4),
}


def most_probable_score(row: pd.Series) -> str:
    best = max(SCORE_COLS_MAP, key=lambda c: row[c])
    h, a = SCORE_COLS_MAP[best]
    return f"{h}-{a}"


def expected_goals(row: pd.Series) -> tuple[float, float]:
    total = sum(row[c] for c in SCORE_COLS_MAP)
    if total == 0:
        return 0.0, 0.0
    xg_h = sum(SCORE_COLS_MAP[c][0] * row[c] for c in SCORE_COLS_MAP) / total
    xg_a = sum(SCORE_COLS_MAP[c][1] * row[c] for c in SCORE_COLS_MAP) / total
    return round(xg_h, 2), round(xg_a, 2)


def result_of(score: str) -> str | None:
    if pd.isna(score) or str(score).strip() == '':
        return None
    parts = str(score).strip().split('-')
    if len(parts) != 2:
        return None
    try:
        h, a = int(parts[0]), int(parts[1])
    except ValueError:
        return None
    return 'H' if h > a else ('A' if h < a else 'D')


def bolao_points(predicted: str, actual: str) -> int | None:
    """3 pts for exact score, 1 pt for correct result."""
    if pd.isna(predicted) or pd.isna(actual) or str(predicted).strip() == '' or str(actual).strip() == '':
        return None
    if str(predicted).strip() == str(actual).strip():
        return 3
    if result_of(predicted) == result_of(actual) and result_of(predicted) is not None:
        return 1
    return 0

In [ ]:
df_grupos = pd.read_csv(DATA_DIR / 'probs_fase_de_grupos.csv')
df_chaveamento = pd.read_csv(DATA_DIR / 'chaveamento_probs.csv')
df_summary = pd.read_csv(DATA_DIR / 'summary.csv')

# Enrich group stage data
df_grupos['placar_modelo'] = df_grupos.apply(most_probable_score, axis=1)
xg_vals = df_grupos.apply(expected_goals, axis=1)
df_grupos['xg_casa'] = [x[0] for x in xg_vals]
df_grupos['xg_fora'] = [x[1] for x in xg_vals]

# Sort by date then group
df_grupos['_date_sort'] = pd.to_datetime('2026/' + df_grupos['date'], format='%Y/%d/%m', errors='coerce')
df_grupos = df_grupos.sort_values(['_date_sort', 'group']).reset_index(drop=True)
df_grupos.insert(0, 'jogo', range(1, len(df_grupos) + 1))

# Build clean display table for group stage
DISPLAY_COLS = ['jogo', 'group', 'date', 'home_team', 'away_team',
                'home_win', 'draw', 'away_win',
                'placar_modelo', 'xg_casa', 'xg_fora']
RENAME = {
    'jogo': '#', 'group': 'Grupo', 'date': 'Data',
    'home_team': 'Casa', 'away_team': 'Fora',
    'home_win': 'P(Casa)%', 'draw': 'P(Empate)%', 'away_win': 'P(Fora)%',
    'placar_modelo': 'Modelo', 'xg_casa': 'xG Casa', 'xg_fora': 'xG Fora',
}
df_display = df_grupos[DISPLAY_COLS].rename(columns=RENAME)

print(f"Jogos na fase de grupos: {len(df_grupos)}")
print(f"Grupos: {sorted(df_grupos['group'].unique())}")

---
## Fase de Grupos — Todos os 72 Jogos

Colunas:
- **P(Casa/Empate/Fora)** → probabilidade do modelo (%) para cada resultado
- **Modelo** → placar com maior probabilidade individual segundo as simulações
- **xG** → gols esperados (média ponderada pelas probabilidades de placar)

In [ ]:
for grupo in sorted(df_grupos['group'].unique()):
    sub = df_display[df_display['Grupo'] == grupo].copy()
    print(f"\n{'='*80}")
    print(f"  GRUPO {grupo}")
    print('='*80)
    display(
        sub.drop(columns='Grupo').style
        .format({'P(Casa)%': '{:.1f}', 'P(Empate)%': '{:.1f}', 'P(Fora)%': '{:.1f}',
                 'xG Casa': '{:.2f}', 'xG Fora': '{:.2f}'})
        .background_gradient(subset=['P(Casa)%', 'P(Fora)%'], cmap='RdYlGn', axis=None)
        .set_properties(**{'text-align': 'center'})
        .hide(axis='index')
    )

---
## Probabilidades Gerais do Torneio

Probabilidade (%) de cada seleção alcançar cada fase, segundo 100 mil simulações.

In [ ]:
df_sum_display = df_summary[[
    'team', 'champion', 'final', 'semifinals', 'quarterfinals',
    'round_of_16', 'round_of_32'
]].rename(columns={
    'team': 'Seleção', 'champion': 'Campeão%', 'final': 'Final%',
    'semifinals': 'Semi%', 'quarterfinals': 'Quartas%',
    'round_of_16': 'Oitavas%', 'round_of_32': 'R32%',
})

display(
    df_sum_display.style
    .format({c: '{:.1f}' for c in df_sum_display.columns if '%' in c})
    .background_gradient(subset=['Campeão%'], cmap='YlOrRd')
    .bar(subset=['Final%', 'Semi%', 'Quartas%'], color='#6baed6')
    .set_caption('Top seleções — probabilidades por fase (%)')
    .hide(axis='index')
)

---
## Top 16 — Probabilidade de Título

In [ ]:
top16 = df_summary.nlargest(16, 'champion')
colors = ['#009C3B' if t == 'Brasil' else '#003087' if t == 'Argentina' else '#CE1124'
          if t == 'Inglaterra' else '#c0392b' if t == 'Espanha' else '#003189'
          if t == 'França' else '#4a90d9'
          for t in top16['team']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top16['team'][::-1], top16['champion'][::-1], color=colors[::-1], edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, top16['champion'][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Probabilidade de Título (%)', fontsize=11)
ax.set_title('Copa do Mundo 2026 — Top 16 Favoritos', fontsize=13, fontweight='bold')
ax.set_xlim(0, top16['champion'].max() * 1.2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Fase Eliminatória — Chaveamento Previsto pelo Modelo

> As equipes abaixo são as mais prováveis segundo as simulações. O chaveamento real depende dos resultados da fase de grupos.

In [ ]:
ROUND_LABEL_PT = {
    'R32': 'Round of 32', 'Oitavas': 'Oitavas de Final',
    'Quartas': 'Quartas de Final', 'Semifinal': 'Semifinal',
    'Final': 'Final', '3º Lugar': '3º Lugar',
}

for fase in df_chaveamento['round_label'].unique():
    sub = df_chaveamento[df_chaveamento['round_label'] == fase].copy()
    label = ROUND_LABEL_PT.get(fase, fase)
    print(f"\n{'─'*70}")
    print(f"  {label.upper()}")
    print('─'*70)
    for _, row in sub.iterrows():
        winner_tag = '← PREVISTO' if row.get('winner') else ''
        home_bold = '**' if row.get('winner') == 'home' else ''
        away_bold = '**' if row.get('winner') == 'away' else ''
        print(
            f"  Jogo {int(row['order']):>2} │ {row['home_team']:<30} {row['prob_home']:>5.1f}%  vs  "
            f"{row['prob_away']:>5.1f}%  {row['away_team']:<30}"
        )

---
## Bolão — Palpites dos Participantes

### Como preencher
1. Edite o dicionário `PARTICIPANTES` abaixo com os nomes dos participantes
2. Preencha o dicionário `PALPITES` com os placares previstos no formato `'gols_casa-gols_fora'` (ex: `'2-1'`)
3. Execute a célula para ver a tabela de palpites
4. Quando os resultados saírem, preencha `RESULTADOS_REAIS` e execute a célula de pontuação

**Fase de grupos:** use o número do jogo (coluna `#` na tabela acima)  
**Fase eliminatória:** os jogos continuam sendo numerados a partir do 73

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDITE AQUI: nomes dos participantes e seus palpites
# Formato: { numero_do_jogo: 'gols_casa-gols_fora' }
# Use '' para jogos ainda não preenchidos
# ─────────────────────────────────────────────────────────────────────────────

PARTICIPANTES: list[str] = [
    'João',
    'Maria',
    'Pedro',
    # Adicione mais participantes aqui
]

PALPITES: dict[str, dict[int, str]] = {
    'João': {
        # Exemplo: jogo 1 = México vs África do Sul (11/06)
        1: '2-0',   # México vs África do Sul
        2: '1-1',   # Coreia do Sul vs Tchéquia
        3: '2-1',   # Brasil vs Marrocos
        # ... continue para todos os jogos
    },
    'Maria': {
        1: '1-0',
        2: '2-0',
        3: '1-0',
    },
    'Pedro': {
        1: '1-1',
        2: '1-2',
        3: '0-1',
    },
}

RESULTADOS_REAIS: dict[int, str] = {
    # Preencha conforme os resultados acontecem
    # Exemplo: 1: '2-0',
}

# ─────────────────────────────────────────────────────────────────────────────
# Monta tabela de palpites
base_cols = df_display[['#', 'Data', 'Casa', 'Fora', 'Modelo']].copy()
base_cols['Resultado Real'] = base_cols['#'].map(RESULTADOS_REAIS).fillna('')
for p in PARTICIPANTES:
    base_cols[p] = base_cols['#'].map(PALPITES.get(p, {})).fillna('')

# Apenas jogos com pelo menos um palpite
has_any = base_cols[PARTICIPANTES].apply(lambda r: any(v != '' for v in r), axis=1)
df_bolao_view = base_cols[has_any].reset_index(drop=True)
print(f"Jogos com palpites registrados: {len(df_bolao_view)}")
display(df_bolao_view.style.hide(axis='index'))

---
## Pontuação — Classificação do Bolão

In [ ]:
if not RESULTADOS_REAIS:
    print("Nenhum resultado real registrado ainda. Preencha RESULTADOS_REAIS para ver a pontuação.")
else:
    scores: dict[str, dict[str, int]] = {p: {'Exact': 0, 'Result': 0, 'Total': 0} for p in PARTICIPANTES}

    for jogo, resultado in RESULTADOS_REAIS.items():
        for p in PARTICIPANTES:
            palpite = PALPITES.get(p, {}).get(jogo, '')
            pts = bolao_points(palpite, resultado)
            if pts is None:
                continue
            scores[p]['Total'] += pts
            if pts == 3:
                scores[p]['Exact'] += 1
            elif pts == 1:
                scores[p]['Result'] += 1

    df_scores = pd.DataFrame([
        {'Participante': p, 'Placares Exatos': v['Exact'], 'Resultados Certos': v['Result'], 'Total de Pontos': v['Total']}
        for p, v in scores.items()
    ]).sort_values('Total de Pontos', ascending=False).reset_index(drop=True)
    df_scores.index = df_scores.index + 1
    df_scores.index.name = 'Pos'

    print(f"Jogos apurados: {len(RESULTADOS_REAIS)}")
    display(
        df_scores.style
        .background_gradient(subset=['Total de Pontos'], cmap='YlGn')
        .format({'Total de Pontos': '{:d}', 'Placares Exatos': '{:d}', 'Resultados Certos': '{:d}'})
        .set_caption('Classificação do Bolão')
    )

---
## Exportar CSV do Bolão

Gera um arquivo `data/bolao.csv` com todos os jogos, probabilidades do modelo e palpites preenchidos.

In [ ]:
# Build full export table (72 group stage games)
export_cols = ['jogo', 'group', 'date', 'home_team', 'away_team',
               'home_win', 'draw', 'away_win', 'placar_modelo', 'xg_casa', 'xg_fora']
df_export = df_grupos[export_cols].rename(columns={
    'jogo': 'jogo', 'group': 'grupo', 'date': 'data',
    'home_team': 'time_casa', 'away_team': 'time_fora',
    'home_win': 'prob_casa_%', 'draw': 'prob_empate_%', 'away_win': 'prob_fora_%',
    'placar_modelo': 'placar_modelo', 'xg_casa': 'xg_casa', 'xg_fora': 'xg_fora',
})
df_export['resultado_real'] = df_export['jogo'].map(RESULTADOS_REAIS).fillna('')
for p in PARTICIPANTES:
    df_export[f'palpite_{p}'] = df_export['jogo'].map(PALPITES.get(p, {})).fillna('')

# Append knockout stage rows (with TBD teams)
knockout_rows = []
for _, row in df_chaveamento.iterrows():
    knockout_rows.append({
        'jogo': 72 + int(row['order']),
        'grupo': row['round_label'],
        'data': '',
        'time_casa': row['home_team'],
        'time_fora': row['away_team'],
        'prob_casa_%': round(row['prob_home'], 1),
        'prob_empate_%': '',
        'prob_fora_%': round(row['prob_away'], 1),
        'placar_modelo': '',
        'xg_casa': '',
        'xg_fora': '',
        'resultado_real': '',
        **{f'palpite_{p}': '' for p in PARTICIPANTES},
    })

df_knockout_export = pd.DataFrame(knockout_rows)
df_full_export = pd.concat([df_export, df_knockout_export], ignore_index=True)

out_path = DATA_DIR / 'bolao.csv'
df_full_export.to_csv(out_path, index=False)
print(f"Exportado: {out_path}")
print(f"Total de linhas: {len(df_full_export)} (72 fase de grupos + {len(knockout_rows)} fase eliminatória)")
df_full_export.head(10)